# **01 - Ingest**

This notebook loads the raw Big Data Bowl 2023 csvs - games, plays, players, PFF scouting data, and the weekly tracking files - and prepares a single filtered tracking dataset scoped to pass-blocking plays. It filters plays to dropback outcomes relevant to pass protection, scopes PFF block types by dropping non-blocks and flagging shared or handed-off blocks rather than discarding them, trims each play's tracking data to the snap-to-outcome frame window, and joins the PFF scouting context onto that trimmed tracking data before combining all eight weeks into a single DataFrame, `tracking_df`.

In [ ]:
import pandas as pd
from pathlib import Path
raw_dir = Path("~/Desktop/Projects/Trenches/nfl-big-data-bowl-2023")
weeks = range(1,9)
drop_block_types = {"NB"}
assisted_block_types = {"SW", "PT", "BH", "CH"}
end_events = {"pass_forward", "qb_sack", "qb_strip_sack", "fumble"}

In [ ]:
games = pd.read_csv("~/Desktop/Projects/Trenches/nfl-big-data-bowl-2023/games.csv")
plays = pd.read_csv("~/Desktop/Projects/Trenches/nfl-big-data-bowl-2023/plays.csv")
players = pd.read_csv("~/Desktop/Projects/Trenches/nfl-big-data-bowl-2023/players.csv")
pff = pd.read_csv("~/Desktop/Projects/Trenches/nfl-big-data-bowl-2023/pffScoutingData.csv")

In [ ]:
#scrambles (R) excluded, not a fair blocking rep
valid_results = {"C", "I", "S", "IN"}
valid_plays = plays[plays["passResult"].isin(valid_results)].copy()
print(f"Plays retained: {len(valid_plays):,} of {len(plays):,} ({len(valid_plays)/len(plays):.1%})")

In [ ]:
#NB (no block) dropped outright. SW/PT/BH/CH get kept but flagged since they represent shared/handed-off blocking responsibility
pff_scoped = pff[~pff["pff_blockType"].isin(drop_block_types)].copy()
pff_scoped["assisted_block"] = pff_scoped["pff_blockType"].isin(assisted_block_types)
print(f"PFF rows retained: {len(pff_scoped):,} of {len(pff):,} ({len(pff_scoped)/len(pff):.1%})")

In [ ]:
def get_frame_window(week_df: pd.DataFrame) -> pd.DataFrame:
    trimmed_plays = []
    for (game_id, play_id), play_frames in week_df.groupby(["gameId", "playId"]):
        events = play_frames[["frameId", "event"]].drop_duplicates()
        snap_rows = events[events["event"] == "ball_snap"]
        if snap_rows.empty:
            continue
        start_frame = snap_rows["frameId"].min()
        end_rows = events[events["event"].isin(end_events)]
        end_frame = end_rows["frameId"].min() if not end_rows.empty else play_frames["frameId"].max()
        window = play_frames[
            (play_frames["frameId"] >= start_frame) & (play_frames["frameId"] <= end_frame)
        ]
        trimmed_plays.append(window)

    if not trimmed_plays:
        return pd.DataFrame(columns=week_df.columns)

    return pd.concat(trimmed_plays, ignore_index=True)

In [ ]:
def process_week(week_num: int) -> pd.DataFrame:
    week_df = pd.read_csv(raw_dir / f"week{week_num}.csv")
    week_df = week_df.merge(
        valid_plays[["gameId", "playId"]], on = ["gameId", "playId"], how = "inner"
    )
    week_df = get_frame_window(week_df)
    week_df = week_df.merge(
        pff_scoped[
            ["gameId", "playId", "nflId", "pff_role", "pff_blockType",
             "assisted_block", "pff_nflIdBlockedPlayer", "pff_sackAllowed",
             "pff_hurryAllowed", "pff_hitAllowed", "pff_beatenByDefender"]
        ],
        on=["gameId", "playId", "nflId"],
        how="inner",
    )
    week_df["week"] = week_num
    return week_df

In [ ]:
week_frames = [process_week(w) for w in weeks]
tracking_df = pd.concat(week_frames, ignore_index=True)

print(f"Total filtered tracking rows: {len(tracking_df):,}")
print(f"Assisted blocks flagged: {tracking_df['assisted_block'].sum():,}")